# Day 17: DevOps and Cloud Fundamentals
**CI/CD concepts, IaaS, PaaS and SaaS, Microsoft Azure and Azure DevOps**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup
We cannot create cloud resources from this notebook, so we simulate a CI pipeline locally and prepare the files you will push to Azure DevOps tomorrow. Azure CLI commands are shown for Cloud Shell.

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q pytest pyflakes pyyaml

# ===== Option B: VS Code / local =====
# pip install pytest pyflakes pyyaml   (in your activated venv)
import os
os.makedirs("ci_demo", exist_ok=True)
os.chdir("ci_demo")
print("Working in", os.getcwd())

## 2. A tiny project to build and test

In [ ]:
%%writefile calc.py
def add(a, b):
    return a + b

def discount(price, pct):
    if not 0 <= pct <= 100:
        raise ValueError("pct must be 0 to 100")
    return round(price * (1 - pct / 100), 2)

In [ ]:
%%writefile test_calc.py
import pytest
from calc import add, discount

def test_add():
    assert add(2, 3) == 5

def test_discount():
    assert discount(200, 10) == 180.0

def test_bad_discount():
    with pytest.raises(ValueError):
        discount(100, 120)

In [ ]:
%%writefile requirements.txt
pytest
pyflakes

## 3. A local CI simulator
Each step is a shell command. The pipeline stops at the first failure, just like Azure Pipelines.

In [ ]:
import subprocess, time, zipfile, glob, shutil, os

def run_pipeline(steps):
    shutil.rmtree("__pycache__", ignore_errors=True)   # a CI agent always starts clean, so do we
    env = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1"}
    start = time.perf_counter()
    for i, (name, cmd) in enumerate(steps, 1):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
        status = "OK" if r.returncode == 0 else "FAILED"
        print(f"[{i}/{len(steps)}] {name:<14} ... {status}")
        if r.returncode != 0:
            print((r.stdout + r.stderr)[-800:])
            print("Pipeline stopped. Fix the problem and push again.")
            return False
    print(f"Pipeline succeeded in {time.perf_counter() - start:.1f} s")
    return True

def package():
    with zipfile.ZipFile("app.zip", "w") as z:
        for f in glob.glob("*.py") + ["requirements.txt"]:
            if not f.startswith("test_"):
                z.write(f)
    return "app.zip"

STEPS = [
    ("Lint", "python -m pyflakes calc.py"),
    ("Unit tests", "python -m pytest -q"),
]
if run_pipeline(STEPS):
    print("Artifact:", package())

### Break it on purpose
This cell introduces a bug, runs the pipeline (it fails), then restores the file.

In [ ]:
good = open("calc.py").read()
open("calc.py", "w").write(good.replace("return a + b", "return a - b"))
run_pipeline(STEPS)
open("calc.py", "w").write(good)
print("\nBug fixed. Running again:")
run_pipeline(STEPS)

### TODO 1: Add a step that fails if any Python file is longer than 200 lines, and see it pass.

In [ ]:
# TODO: write your code here

## 4. The pipeline file for Azure DevOps

In [ ]:
%%writefile azure-pipelines.yml
trigger:
  - main

pool:
  vmImage: ubuntu-latest

steps:
  - task: UsePythonVersion@0
    inputs:
      versionSpec: "3.11"
    displayName: Use Python 3.11

  - script: pip install -r requirements.txt
    displayName: Install dependencies

  - script: python -m pyflakes calc.py
    displayName: Lint

  - script: python -m pytest -q --junitxml=test-results.xml
    displayName: Run tests

  - task: PublishTestResults@2
    condition: always()
    inputs:
      testResultsFiles: test-results.xml
    displayName: Publish test results

In [ ]:
import yaml
cfg = yaml.safe_load(open("azure-pipelines.yml"))
print("Trigger:", cfg["trigger"])
print("Agent:", cfg["pool"]["vmImage"])
for i, step in enumerate(cfg["steps"], 1):
    kind = "task" if "task" in step else "script"
    print(f"{i}. {step['displayName']}  ({kind})")

## 5. Cloud models: quick check

In [ ]:
scenarios = {
    "Legacy app that needs a custom Windows driver": "IaaS",
    "FastAPI service the team just wants to deploy": "PaaS",
    "Company email and documents": "SaaS",
    "Managed relational database nobody wants to patch": "PaaS",
    "Full control of a GPU machine with custom drivers": "IaaS",
}
my_answers = {   # TODO: change these to your own answers, then run
    "Legacy app that needs a custom Windows driver": "IaaS",
    "FastAPI service the team just wants to deploy": "PaaS",
    "Company email and documents": "SaaS",
    "Managed relational database nobody wants to patch": "IaaS",
    "Full control of a GPU machine with custom drivers": "IaaS",
}
score = sum(my_answers[s] == m for s, m in scenarios.items())
for s, m in scenarios.items():
    mark = "correct" if my_answers[s] == m else f"expected {m}"
    print(f"- {s}: {my_answers[s]} ({mark})")
print(f"Score: {score}/{len(scenarios)}")

## 6. Azure CLI (run in Azure Cloud Shell)
Open portal.azure.com, click the Cloud Shell icon, choose Bash, and run these. Only use a subscription you are allowed to use, and delete the resource group afterwards.

In [ ]:
# ===== Option A: Azure Cloud Shell (recommended, nothing to install) =====
# az account show -o table
# az group create -n rg-day17-yourname -l centralindia --tags owner=yourname project=training
# az group list -o table
# az group delete -n rg-day17-yourname --yes --no-wait

# ===== Option B: VS Code terminal =====
# Install the Azure CLI from learn.microsoft.com, run: az login
# Then use the same commands as above.
print("Copy the commands above into Cloud Shell or your terminal.")

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)